In [1]:
# ==============================================================================
# NexBank Credit Decision Engine - PyCaret Model Comparison Script
# ==============================================================================
# Skrip eksperimental ini menggunakan AutoML PyCaret untuk melatih dan membandingkan
# berbagai algoritma Machine Learning guna menentukan model terbaik untuk prediksi 
# risiko gagal bayar (loan_status).
# ==============================================================================

import os
import pandas as pd

# Pastikan Anda menginstal PyCaret sebelum menjalankan skrip ini:
# pip install pycaret

try:
    from pycaret.classification import setup, compare_models, pull, save_model, plot_model
except ImportError:
    print("\n[PERINGATAN] Library 'pycaret' belum terinstal!")
    print("Silakan jalankan perintah berikut di terminal Anda terlebih dahulu:")
    print(">>> pip install pycaret\n")
    exit(1)

def load_and_clean_data():
    """Memuat dataset dan melakukan pembersihan anomali data dasar."""
    jalur_pilihan = [
        "../data/loan_data.csv",
        "data/loan_data.csv",
        "loan_data.csv",
        "../data/loan_data_clean.csv",
        "data/loan_data_clean.csv",
        "loan_data_clean.csv"
    ]
    
    df = None
    for path in jalur_pilihan:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Dataset berhasil dimuat dari: {path}")
            break
            
    if df is None:
        raise FileNotFoundError(
            "Gagal memuat dataset! Pastikan file 'loan_data.csv' atau 'loan_data_clean.csv' tersedia."
        )
        
    # Lakukan pembersihan anomali logis dasar (seperti pada preprocessing.py)
    print("Membersihkan data dari anomali logis...")
    emp_col = 'person_emp_exp' if 'person_emp_exp' in df.columns else 'person_emp_length'
    
    if 'person_age' in df.columns:
        df = df[df['person_age'] <= 100]
    if emp_col in df.columns:
        df = df[df[emp_col] <= 60]
    if 'person_age' in df.columns and emp_col in df.columns:
        df = df[df[emp_col] < df['person_age']]
        
    # Hapus baris dengan target kosong jika ada
    df = df.dropna(subset=['loan_status'])
    
    print(f"Dataset siap dengan dimensi: {df.shape[0]} baris, {df.shape[1]} kolom.")
    return df

def main():
    print("=" * 80)
    print(" EKSPERIMEN AUTOML: PERBANDINGAN MODEL MENGGUNAKAN PYCARET")
    print("=" * 80)
    
    # 1. Load data
    try:
        df = load_and_clean_data()
    except Exception as e:
        print(f"Error: {e}")
        return

    # 2. Setup PyCaret Classification Environment
    print("\n[1/4] Mengonfigurasi lingkungan setup PyCaret...")
    
    # PyCaret akan mendeteksi tipe data secara otomatis.
    # Kita bisa mendefinisikan beberapa parameter agar mirip dengan preprocessing kita sendiri.
    clf_setup = setup(
        data=df,
        target='loan_status',         # Kolom target prediksi
        train_size=0.8,               # Rasio data latih (80% Train, 20% Test)
        session_id=42,                # Seed acak agar hasil konsisten
        numeric_imputation='median',  # Imputasi angka kosong dengan median
        categorical_imputation='mode',# Imputasi kategori kosong dengan modus
        normalize=True,               # Standarisasi skala fitur (StandardScaler)
        normalize_method='zscore',    # z-score (mirip StandardScaler scikit-learn)
        verbose=False                 # Sembunyikan output setup yang terlalu panjang
    )
    print("-> Setup berhasil dikonfigurasi!")

    # 3. Bandingkan semua model klasifikasi
    print("\n[2/4] Melatih dan membandingkan seluruh model klasifikasi (Eksperimen AutoML)...")
    print("Proses ini akan melatih belasan model seperti XGBoost, LightGBM, Random Forest, dll.")
    print("Mohon tunggu beberapa saat...\n")
    
    # compare_models() otomatis melatih semua model dan mengurutkannya berdasarkan metrik tertentu (default: Accuracy)
    # Di sini kita urutkan berdasarkan AUC karena kelas data imbalance.
    best_model = compare_models(sort='AUC')
    
    # Mengambil tabel hasil evaluasi dalam bentuk DataFrame
    leaderboard = pull()
    
    print("\n" + "=" * 80)
    print(" HASIL LEADERBOARD PERBANDINGAN MODEL (DIURUTKAN BERDASARKAN AUC)")
    print("=" * 80)
    print(leaderboard)
    print("=" * 80)

    # 4. Visualisasi & Evaluasi Model Terbaik
    print("\n[3/4] Membuat visualisasi performa untuk model terbaik yang terpilih...")
    
    # Simpan plot evaluasi model terbaik ke dalam direktori lokal
    try:
        os.makedirs("plots_pycaret", exist_ok=True)
        
        # Plot Confusion Matrix
        print("- Menyimpan Confusion Matrix...")
        plot_model(best_model, plot='confusion_matrix', save=True)
        if os.path.exists('Confusion Matrix.png'):
            os.rename('Confusion Matrix.png', 'plots_pycaret/pycaret_confusion_matrix.png')
            
        # Plot ROC Curves
        print("- Menyimpan Kurva ROC...")
        plot_model(best_model, plot='auc', save=True)
        if os.path.exists('AUC.png'):
            os.rename('AUC.png', 'plots_pycaret/pycaret_roc_auc.png')
            
        # Plot Feature Importance (jika model mendukung)
        print("- Menyimpan Feature Importance...")
        try:
            plot_model(best_model, plot='feature', save=True)
            if os.path.exists('Feature Importance.png'):
                os.rename('Feature Importance.png', 'plots_pycaret/pycaret_feature_importance.png')
        except Exception:
            print("  (Model terbaik tidak mendukung plotting Feature Importance secara langsung)")
            
        print("-> Semua grafik evaluasi disimpan dalam folder 'plots_pycaret/'.")
    except Exception as e:
        print(f"Gagal menyimpan plot evaluasi: {e}")

    # 5. Menyimpan Model Terbaik
    print("\n[4/4] Menyimpan arsitektur model terbaik ke disk...")
    os.makedirs("models", exist_ok=True)
    save_model(best_model, 'models/best_pycaret_model')
    print("-> Model terbaik berhasil disimpan di 'models/best_pycaret_model.pkl'.")
    
    print("\nEksperimen PyCaret selesai dengan sukses!")

if __name__ == "__main__":
    main()

 EKSPERIMEN AUTOML: PERBANDINGAN MODEL MENGGUNAKAN PYCARET
Dataset berhasil dimuat dari: data/loan_data.csv
Membersihkan data dari anomali logis...
Dataset siap dengan dimensi: 44990 baris, 14 kolom.

[1/4] Mengonfigurasi lingkungan setup PyCaret...
-> Setup berhasil dikonfigurasi!

[2/4] Melatih dan membandingkan seluruh model klasifikasi (Eksperimen AutoML)...
Proses ini akan melatih belasan model seperti XGBoost, LightGBM, Random Forest, dll.
Mohon tunggu beberapa saat...



,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
xgboost,Extreme Gradient Boosting,0.9336,0.9785,0.8030,0.8875,0.8431,0.8011,0.8028,0.1920
catboost,CatBoost Classifier,0.9332,0.9780,0.7912,0.8963,0.8405,0.7985,0.8010,3.6250
lightgbm,Light Gradient Boosting Machine,0.9322,0.9777,0.7879,0.8945,0.8378,0.7951,0.7977,0.2500
rf,Random Forest Classifier,0.9285,0.9737,0.7645,0.8990,0.8262,0.7816,0.7857,0.5160
gbc,Gradient Boosting Classifier,0.9250,0.9720,0.7675,0.8799,0.8198,0.7727,0.7757,0.8630
et,Extra Trees Classifier,0.9171,0.9662,0.7535,0.8567,0.8017,0.7496,0.7521,0.4310
ada,Ada Boost Classifier,0.9140,0.9644,0.7709,0.8300,0.7993,0.7446,0.7455,0.2950
lr,Logistic Regression,0.8959,0.9541,0.7476,0.7759,0.7613,0.6948,0.6951,0.8720
svm,SVM - Linear Kernel,0.8898,0.9503,0.7271,0.7661,0.7454,0.6752,0.6761,0.1220
ridge,Ridge Classifier,0.8948,0.9503,0.6920,0.8072,0.7450,0.6792,0.6825,0.0970



 HASIL LEADERBOARD PERBANDINGAN MODEL (DIURUTKAN BERDASARKAN AUC)
                                    Model  Accuracy     AUC  Recall   Prec.  \
xgboost         Extreme Gradient Boosting    0.9336  0.9785  0.8030  0.8875   
catboost              CatBoost Classifier    0.9332  0.9780  0.7912  0.8963   
lightgbm  Light Gradient Boosting Machine    0.9322  0.9777  0.7879  0.8945   
rf               Random Forest Classifier    0.9285  0.9737  0.7645  0.8990   
gbc          Gradient Boosting Classifier    0.9250  0.9720  0.7675  0.8799   
et                 Extra Trees Classifier    0.9171  0.9662  0.7535  0.8567   
ada                  Ada Boost Classifier    0.9140  0.9644  0.7709  0.8300   
lr                    Logistic Regression    0.8959  0.9541  0.7476  0.7759   
svm                   SVM - Linear Kernel    0.8898  0.9503  0.7271  0.7661   
ridge                    Ridge Classifier    0.8948  0.9503  0.6920  0.8072   
lda          Linear Discriminant Analysis    0.8941  0.9503  0.7

- Menyimpan Kurva ROC...


- Menyimpan Feature Importance...


-> Semua grafik evaluasi disimpan dalam folder 'plots_pycaret/'.

[4/4] Menyimpan arsitektur model terbaik ke disk...
Transformation Pipeline and Model Successfully Saved
-> Model terbaik berhasil disimpan di 'models/best_pycaret_model.pkl'.

Eksperimen PyCaret selesai dengan sukses!
